# Stage C 03o — E100 analysis and matched-control allocation gate


In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='df20b8614ca3165d365dd945948ca24d9495d27e'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
SOURCE_ROOT='/content/drive/MyDrive/bacteria_titan_v1_ecoli_related_15gbp'
DATASET_NAME='nonoverlap_6mer_v1'
TAXONOMY_MANIFEST=f'{DRIVE_ROOT}/stage_c_dataset/manifests/accession_manifest.parquet'
ACCESSION_MANIFEST=TAXONOMY_MANIFEST
ANI_MEMBERSHIP=f'{DRIVE_ROOT}/stage_c_dataset/manifests/ani99_membership.parquet'
ANI_PAIRS=f'{DRIVE_ROOT}/inputs/ecoli_skani_triangle.tsv'
NCBI_ZIP_DIR=f'{SOURCE_ROOT}/raw/ncbi_dataset_zips'
RUN_NAME='c20_v3_medium_adaptive_e100_increment'
RUN_ID='medium_adaptive_e100_analysis_v1'
STAGE='e100'


In [ ]:
from pathlib import Path
from google.colab import drive
import json, shutil, subprocess, sys
mount=Path('/content/drive')
if not (mount/'MyDrive').is_dir(): drive.mount(str(mount),timeout_ms=120000)
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
dataset=Path(DRIVE_ROOT)/'stage_c_dataset/ordered_streams'/DATASET_NAME
panels=Path(DRIVE_ROOT)/'study/stage_c_ecoli_medium_deep_memory_v3/panels'
PROTOCOL=repo/'studies/stage_c_ecoli_medium_deep_memory_v3/protocol.json'
STUDY_ROOT=Path(DRIVE_ROOT)/'study/stage_c_ecoli_medium_deep_memory_v3'
subprocess.run(['seqtrainer-titans-stage-c-study','initialize','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)
def run_logged(root,label,command):
    root.mkdir(parents=True,exist_ok=True)
    try:
        subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',str(root),'--label',label,'--repo',str(repo),'--',*command],check=True)
    except subprocess.CalledProcessError:
        for path in (root/'FAILED.txt',root/'logs'/f'{label}.log'):
            if path.exists(): print(path.read_text(errors='replace')[-20000:])
        raise
def record_once(run_id,artifact,tier):
    marker=STUDY_ROOT/'record_markers'/f'{run_id}.json'
    if marker.exists():
        print('Ledger record already exists:',marker); return
    subprocess.run(['seqtrainer-titans-stage-c-study','record','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--run-id',run_id,'--evidence-tier',tier,'--artifact',str(artifact)],check=True)
    marker.parent.mkdir(parents=True,exist_ok=True)
    marker.write_text(json.dumps({'run_id':run_id,'artifact':str(artifact)},indent=2)+'\n')


In [ ]:
import torch
root=Path(DRIVE_ROOT)/'runs'/RUN_NAME; checkpoint=root/'latest.pt'; out=root/'scale_analysis_v1'
run_logged(out,'evaluate_e100',['seqtrainer-titans-stage-c-evaluate','--dataset-dir',str(dataset),'--panel-manifest',str(panels/'validation.json'),'--run',f'adaptive={checkpoint}','--output-dir',str(out/'evaluation'),'--split','val','--comparison-mode','partial','--device','cuda','--protocol',str(PROTOCOL),'--run-id',RUN_ID])
run_logged(out,'memory_behavior_e100',['seqtrainer-titans-stage-c-memory-behavior','--checkpoint',str(checkpoint),'--output',str(out/'memory_behavior.json'),'--pairs','64','--device','cuda','--protocol',str(PROTOCOL),'--run-id',RUN_ID])
if not shutil.which('prodigal'):
    subprocess.run(['apt-get','update'],check=True); subprocess.run(['apt-get','install','-y','prodigal'],check=True)
run_logged(out,'generation_e100',['seqtrainer-titans-stage-c-generate','--dataset-dir',str(dataset),'--panel-manifest',str(panels/'validation.json'),'--taxonomy-manifest',TAXONOMY_MANIFEST,'--checkpoint',str(checkpoint),'--output-dir',str(out/'generation_t0p6'),'--split','val','--species','Escherichia coli','--prompts','4','--prompt-tokens','128','--new-tokens','1024','--temperatures','0.6','--top-k','1024','--top-p','0.99','--device','cuda','--memory-mode','adaptive','--prodigal',shutil.which('prodigal'),'--protocol',str(PROTOCOL),'--run-id',RUN_ID])
e25=Path(DRIVE_ROOT)/'runs/c19_v3_medium_adaptive_e25/scale_analysis_v1'
subprocess.run(['seqtrainer-titans-stage-c-scale-gate','--stage',STAGE,'--baseline-evaluation',str(e25/'evaluation/evaluation.json'),'--baseline-name','adaptive','--candidate-evaluation',str(out/'evaluation/evaluation.json'),'--candidate-name','adaptive','--memory-behavior',str(out/'memory_behavior.json'),'--baseline-generation',str(e25/'generation_t0p6/generation_evaluation.json'),'--candidate-generation',str(out/'generation_t0p6/generation_evaluation.json'),'--output-dir',str(out/'gate')],check=True)
record_once(RUN_ID,out,'exploratory')
print((out/'gate/SCALE_GATE.md').read_text())
print('A PROCEED result authorizes planning matched no-memory/frozen/second-seed controls; it does not itself prove a memory benefit.')
print('SHARE THIS DIRECTORY:',out)
